<a href="https://colab.research.google.com/github/Charlestonjm21/MotionPrivacyResearch/blob/Load-dataset/NTU_RGBD_STGCN_Benchmark_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MotionPM Project — ST-GCN Benchmark on NTU RGB+D 60
**Purpose:** Reproduce the published ST-GCN baseline results on NTU RGB+D (60-class, skeleton modality only).  
These results serve as the **privacy threat baseline** for the MotionPM privacy measurement framework.

**Expected benchmark results (Yan et al., AAAI 2018):**
| Evaluation Protocol | ST-GCN Top-1 Accuracy |
|---|---|
| Cross-Subject (X-Sub) | 80.7% |
| Cross-View (X-View) | 88.9% |

---
> ⚠️ **Before running:** Set runtime to GPU — `Runtime → Change runtime type → T4 GPU`

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

SKELETON_DATA_PATH = '/content/drive/MyDrive/MotionPrivacyResearch/datasets/nturgbd_skeletons_s001_to_s017/nturgb+d_skeletons'

# Don't list the folder — just check it exists and sample one known file
# os.listdir() on 56K files crashes Drive; we avoid it entirely
test_file = os.path.join(SKELETON_DATA_PATH, 'S015C001P017R001A031.skeleton')

if os.path.exists(SKELETON_DATA_PATH):
    print(f"✅ Folder exists: {SKELETON_DATA_PATH}")
    if os.path.exists(test_file):
        print(f"✅ Sample file confirmed: S015C001P017R001A031.skeleton")
        print(f"✅ Drive connection looks good. Proceeding...")
    else:
        print(f"⚠️  Folder exists but sample file not found — check folder contents")
else:
    print(f"❌ Path not found. Check Drive mount.")

✅ Folder exists: /content/drive/MyDrive/MotionPrivacyResearch/datasets/nturgbd_skeletons_s001_to_s017/nturgb+d_skeletons
✅ Sample file confirmed: S015C001P017R001A031.skeleton
✅ Drive connection looks good. Proceeding...


## Step 2 — Clone the ST-GCN Repository

In [ ]:
%cd /content
!git clone https://github.com/yysijie/st-gcn.git
%cd st-gcn
print("✅ Repository cloned.")

/content
fatal: destination path 'st-gcn' already exists and is not an empty directory.
/content/st-gcn
✅ Repository cloned.


## Step 3 — Install Dependencies and Apply Patches
We install all required packages and patch one import in the repo that breaks
skeleton-only pipelines (it tries to import a video utility we do not need).

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q pyyaml tensorboardX scikit-video

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU")

PyTorch version: 2.10.0+cu128
CUDA available:  True
GPU:             Tesla T4


In [ ]:
# Patch tools/utils/__init__.py — comment out the video import
# which is not needed for skeleton-only processing and causes an import error
init_path = '/content/st-gcn/tools/utils/__init__.py'

with open(init_path, 'r') as f:
    content = f.read()

patched = content.replace(
    'from . import video',
    '# from . import video  # patched: not needed for skeleton-only pipeline'
)

with open(init_path, 'w') as f:
    f.write(patched)

print("Patched __init__.py:")
print(open(init_path).read())
print("\n✅ Patch applied.")

Patched __init__.py:
# # # from . import video  # patched: not needed for skeleton-only pipeline  # patched: not needed for skeleton-only pipeline  # patched: not needed for skeleton-only pipeline
from . import openpose
from . import visualization

✅ Patch applied.


## Step 4 — Copy Skeleton Files to Local Colab Storage
Reading 56,880 files directly from Google Drive is very slow due to Drive I/O limits.
We copy them once to Colab's local disk where preprocessing runs much faster.

> ⏳ This takes **10–15 minutes**. Run it and let it finish before moving on.

In [ ]:
import shutil, os, time

dst = '/content/nturgb+d_skeletons'
src = '/content/drive/MyDrive/MotionPrivacyResearch/datasets/nturgbd_skeletons_s001_to_s017/nturgb+d_skeletons'

# Remove the incomplete copy
if os.path.exists(dst):
    print(f"Removing incomplete copy ({len(os.listdir(dst))} files)...")
    shutil.rmtree(dst)
    print("✅ Removed.")

# Fresh copy
print("\nStarting fresh copy from Drive → local storage...")
print("Do not interrupt this cell.\n")
start = time.time()
shutil.copytree(src, dst)
elapsed = (time.time() - start) / 60
count = len(os.listdir(dst))
print(f"\n✅ Done in {elapsed:.1f} min. Files copied: {count}")

if count < 56000:
    print(f"⚠️  Expected ~56,880 files but got {count}. Drive connection may be unstable.")
else:
    print(f"✅ File count correct. Ready for preprocessing.")

Removing incomplete copy (4407 files)...
✅ Removed.

Starting fresh copy from Drive → local storage...
Do not interrupt this cell.



## Step 5 — Preprocess Skeleton Data
Converts raw `.skeleton` files into the numpy arrays ST-GCN expects,
and generates train/test splits for both evaluation protocols.

> ⏳ This takes **20–40 minutes**. Let it run to completion.

In [ ]:
import os

# Create output directory
!mkdir -p /content/st-gcn/data/NTU-RGB-D

# Download the list of samples with missing skeletons (302 files to ignore)
!wget -q -O /content/st-gcn/data/NTU-RGB-D/samples_with_missing_skeletons.txt \
  https://raw.githubusercontent.com/shahroudy/NTURGB-D/master/Matlab/NTU_RGBD_samples_with_missing_skeletons.txt
print("✅ Missing skeletons list downloaded.")

In [ ]:
%cd /content/st-gcn

# Run preprocessing — reads local skeleton files and outputs .npy + .pkl splits
!python tools/ntu_gendata.py \
    --data_path /content/nturgb+d_skeletons \
    --ignored_sample_path /content/st-gcn/data/NTU-RGB-D/samples_with_missing_skeletons.txt \
    --out_folder /content/st-gcn/data/NTU-RGB-D/

print("\n✅ Preprocessing complete.")

In [ ]:
# Verify all output files were created
import os

expected_files = [
    'data/NTU-RGB-D/xsub/train_data.npy',
    'data/NTU-RGB-D/xsub/train_label.pkl',
    'data/NTU-RGB-D/xsub/val_data.npy',
    'data/NTU-RGB-D/xsub/val_label.pkl',
    'data/NTU-RGB-D/xview/train_data.npy',
    'data/NTU-RGB-D/xview/train_label.pkl',
    'data/NTU-RGB-D/xview/val_data.npy',
    'data/NTU-RGB-D/xview/val_label.pkl',
]

all_ok = True
for f in expected_files:
    path = f'/content/st-gcn/{f}'
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    status = f'✅ {size_mb:.1f} MB' if exists else '❌ MISSING'
    print(f"{status}  {f}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ All files present. Ready for evaluation.")
else:
    print("\n❌ Some files missing. Check preprocessing output above for errors.")

## Step 6 — Save Processed Data to Drive
Save the `.npy` files back to Drive now so if Colab disconnects,
you never have to redo the 40-minute preprocessing again.

In [ ]:
import shutil, os

SAVE_PATH = '/content/drive/MyDrive/MotionPrivacyResearch/datasets/processed_npy'
os.makedirs(SAVE_PATH, exist_ok=True)

for split in ['xsub', 'xview']:
    src = f'/content/st-gcn/data/NTU-RGB-D/{split}'
    dst = f'{SAVE_PATH}/{split}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"✅ Saved {split} → {dst}")
    else:
        print(f"❌ Source not found: {src} — run preprocessing first.")

print("\nNext session: load directly from Drive .npy files, skip preprocessing.")

## Step 7 — Download Pretrained ST-GCN Weights
The authors provide pretrained weights — no need to train from scratch.
Training would take ~3 days; evaluation with pretrained weights takes ~10 minutes.

In [ ]:
%cd /content/st-gcn
!bash tools/get_models.sh

import os
print("\nDownloaded model files:")
for m in os.listdir('/content/st-gcn/models'):
    if m.endswith('.pt'):
        size_mb = os.path.getsize(f'/content/st-gcn/models/{m}') / 1e6
        print(f"  ✅ {m}  ({size_mb:.1f} MB)")

## Step 8 — Evaluate: Cross-Subject Protocol
**Published target: 80.7% Top-1 accuracy**

Split by performer ID. The model has never seen the test subjects during training.
This is the harder, more realistic evaluation protocol.

In [ ]:
%cd /content/st-gcn

print("ST-GCN Evaluation — Cross-Subject (X-Sub)")
print("Published target: 80.7%\n")

!python main.py --config config/st_gcn/nturgbd-cross-subject/test.yaml \
    --weights models/st_gcn.nturgbd-cross-subject.pt \
    --test-batch-size 64 \
    --device 0

## Step 9 — Evaluate: Cross-View Protocol
**Published target: 88.9% Top-1 accuracy**

Split by camera angle. Camera 1 = test; Cameras 2 & 3 = train.
Tests generalization across viewpoints.

In [ ]:
%cd /content/st-gcn

print("ST-GCN Evaluation — Cross-View (X-View)")
print("Published target: 88.9%\n")

!python main.py --config config/st_gcn/nturgbd-cross-view/test.yaml \
    --weights models/st_gcn.nturgbd-cross-view.pt \
    --test-batch-size 64 \
    --device 0

## Step 10 — Record Results

In [ ]:
# Fill in your observed results after running Steps 8 and 9
observed_xsub  = None   # e.g. 80.5  ← replace with your value
observed_xview = None   # e.g. 88.7  ← replace with your value

published_xsub  = 80.7
published_xview = 88.9

print("=" * 58)
print("  MotionPM — ST-GCN Benchmark Results (NTU RGB+D 60)")
print("=" * 58)
print(f"{'Protocol':<20} {'Published':>12} {'Observed':>12} {'Delta':>8}")
print("-" * 58)

for name, pub, obs in [
    ('Cross-Subject', published_xsub,  observed_xsub),
    ('Cross-View',    published_xview, observed_xview)
]:
    if obs is not None:
        delta = obs - pub
        flag = '✅' if abs(delta) < 1.0 else '⚠️ check weights'
        print(f"{name:<20} {pub:>11.1f}% {obs:>11.1f}% {delta:>+7.1f}%  {flag}")
    else:
        print(f"{name:<20} {pub:>11.1f}%  {'(not run yet)':>20}")

print("=" * 58)
print()
print("MotionPM context:")
print("  These numbers = adversary's action recognition accuracy")
print("  on RAW unprotected skeleton data. This is the UPPER")
print("  BOUND in your privacy-utility tradeoff analysis.")
print("  After applying privacy mechanisms, accuracy should drop.")
print("  The size of that drop = your privacy gain metric.")

---
## Troubleshooting

**`CUDA out of memory`** → Change `--test-batch-size 64` to `32` or `16`

**`get_models.sh` fails** → Authors' links may have expired. Manually download `.pt` files from:  
https://drive.google.com/drive/folders/1BNNpKi3UVbBW8ODFJhBPr43JjLsEqNsz  
Place them in `/content/st-gcn/models/`

**Accuracy > 2% below published** → Verify correct weights file is matched to correct protocol (xsub ↔ xview)

**Colab disconnected mid-preprocessing** → Processed `.npy` files are saved to Drive in Step 6.  
Next session: copy them back with `shutil.copytree` into `/content/st-gcn/data/NTU-RGB-D/` and skip Step 5.

---
**Citation:**  
Yan, S., Xiong, Y., & Lin, D. (2018). Spatial temporal graph convolutional networks for skeleton-based action recognition. *AAAI 2018.*